# Train ACT on AlohaTransferCube

This notebook is the Colab version of `train.py` and `evaluate.py` in the mimic-arm repo.

You will:

1. Install LeRobot 0.6.1 with the ALOHA simulator, without throwing away Colab's CUDA PyTorch.
2. Train an ACT policy on `lerobot/aloha_sim_transfer_cube_human` for 8,000 steps.
3. Evaluate it for 20 episodes in `AlohaTransferCube`, save a video of each failure, and watch one.
4. Optionally keep checkpoints on Google Drive, train out to 20,000 steps, look at the size of the network, and compare checkpoints.

Use a **GPU runtime** (Runtime → Change runtime type → T4). A CPU runtime will train, but 8,000 steps will not finish during a free session.


## Why 8,000 steps, not 100,000

LeRobot's default ACT run is **100,000 steps** at batch size 8. The checkpoint Hugging Face published for this task (`lerobot/act_aloha_sim_transfer_cube_human`) was saved at **80,000** steps and is the one that actually hands the cube over.

A free T4 session often ends after a couple of hours, and sometimes sooner. 100,000 steps is an overnight job on a better GPU, not a free-tier notebook.

**8,000 steps** is the compromise used here:

- long enough that the loss should fall (the policy starts to imitate the demonstrations)
- short enough that a T4 can plausibly finish, including the dataset download
- checkpointed every 2,000 steps, so a disconnect keeps a usable folder

Expect a low success rate afterwards. The arms may start reaching. They will not reliably transfer the cube. If the session is still healthy at the end, train longer: resume from `outputs/train/act_aloha_transfer_cube_colab` and raise `--steps`. The loss curve is the signal to watch at this length, not the success rate.

On a CPU smoke-test machine, batch size 2 took about one second per step. A T4 at batch size 8 is much faster per sample, but 8,000 steps plus evaluation is still the bulk of a free session. Plan on roughly one to two hours. That number was not timed on a T4.
The **Train longer** section at the bottom resumes this run instead of starting over. Set `TOTAL_STEPS` there (it starts at 20,000).


In [ ]:
import sys
import torch

print("Python", sys.version.split()[0])
print("torch", torch.__version__, "cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU", torch.cuda.get_device_name(0))
else:
    print("No GPU visible. Switch the runtime to a T4 before training 8000 steps.")

if sys.version_info < (3, 12):
    raise SystemExit(
        "LeRobot 0.6.1 needs Python 3.12 or newer. "
        "This Colab runtime is too old for the pinned install."
    )


## Install

Colab already has a CUDA build of PyTorch matched to its driver. A plain `pip install lerobot` will try to replace it with a CUDA 13 wheel, which a free T4's driver often cannot load. This cell writes a constraints file pinning `torch` and `torchvision` to the versions already imported, then installs LeRobot against that file. Pip's output is shown in full.

`gym-aloha` depends on `dm-control`, which depends on `labmaze`. LabMaze 1.0.6 publishes binary wheels through Python 3.12 only. On this runtime's Python 3.13 the source build needs Bazel and fails. AlohaTransferCube never imports labmaze, so when no binary wheel exists the cell installs a pure-Python placeholder of the same version first.

System packages: EGL for MuJoCo-on-GPU, OSMesa as a CPU fallback.


In [ ]:
import importlib
import os
import subprocess
import sys
from pathlib import Path

import torch
import torchvision

orig_torch = torch.__version__
orig_vision = torchvision.__version__
print("Keeping Colab torch", orig_torch, "vision", orig_vision)

subprocess.check_call(["apt-get", "update", "-qq"])
subprocess.check_call(
    [
        "apt-get", "install", "-y", "-qq",
        "libegl1", "libgl1", "libglib2.0-0", "libosmesa6", "ffmpeg",
    ]
)

# The labmaze helper lives in the repo. Colab downloads only this notebook,
# so clone first when train.py is not already in the working directory.
# Branch or tag: MIMIC_ARM_REF in the environment, otherwise main. The next
# cell sets that name in the notebook and checks the checkout again.
if not Path("train.py").is_file():
    repo = os.environ.get("MIMIC_ARM_REPO", "https://github.com/Barbodmz/mimic-arm")
    ref = os.environ.get("MIMIC_ARM_REF", "main")
    subprocess.check_call(
        ["git", "clone", "--depth", "1", "--branch", ref, repo, "mimic-arm"]
    )
    os.chdir("mimic-arm")

# On Python 3.13 this installs a labmaze==1.0.6 placeholder. On 3.12 it
# leaves the published wheel for the requirements install.
subprocess.check_call([sys.executable, "-m", "mimic_arm.ensure_labmaze"])

constraints = Path("/tmp/mimic-arm-torch-constraints.txt")
constraints.write_text(f"torch=={orig_torch}\ntorchvision=={orig_vision}\n")
print("Torch constraints:")
print(constraints.read_text(), end="")

# numpy and fsspec are intentionally not pinned on this command. LeRobot
# 0.6.1 requires numpy<2.3 and datasets 4.8.5 requires fsspec<=2026.2.0, and
# the resolver downgrades newer copies that shipped with the preinstalled
# torch. Hard-pinning them here is unnecessary; the failure mode on Python
# 3.13 was labmaze's missing wheel, above.
subprocess.check_call(
    [
        sys.executable, "-m", "pip", "install",
        "-c", str(constraints),
        "lerobot[training,aloha,evaluation]==0.6.1",
    ]
)

importlib.invalidate_caches()
# Re-import after the install. A replaced torch module may still be the old one
# until the process restarts, so also ask a fresh interpreter.
probe = subprocess.run(
    [sys.executable, "-c", "import torch; print(torch.__version__); print(torch.cuda.is_available())"],
    check=True,
    capture_output=True,
    text=True,
)
print(probe.stdout)
cuda_ok = probe.stdout.strip().splitlines()[-1] == "True"
if not cuda_ok and "+" in orig_torch:
    cuda_tag = orig_torch.split("+", 1)[1]
    if cuda_tag.startswith("cu"):
        index = f"https://download.pytorch.org/whl/{cuda_tag}"
        print(f"CUDA torch was replaced. Restoring {orig_torch} from {index}")
        subprocess.check_call(
            [
                sys.executable, "-m", "pip", "install", "--force-reinstall",
                f"torch=={orig_torch}",
                f"torchvision=={orig_vision}",
                "--index-url", index,
            ]
        )
        # That index upgrades numpy past 2.2 (LeRobot rejects numpy>=2.3) and
        # fsspec past 2026.2.0 (datasets 4.8.5 rejects that).
        subprocess.check_call(
            [
                sys.executable, "-m", "pip", "install",
                "-c", str(constraints),
                "numpy>=2.0,<2.3",
                "fsspec>=2023.1.0,<=2026.2.0",
            ]
        )

import lerobot
print("lerobot", lerobot.__version__)


## Get the training scripts

`train.py` and `evaluate.py` are the same files you run locally. Colab only downloads this notebook, so the scripts have to be cloned.

`MIMIC_ARM_REF` is the branch or tag that gets cloned. It is `"main"` unless you change it. Opening the notebook from another link does not change the clone: the cell below still uses `MIMIC_ARM_REF`. To reproduce the v1 baseline, set `MIMIC_ARM_REF = "v1-baseline"`. If the environment variable `MIMIC_ARM_REF` is set, that value wins over the line in the cell.

Run this cell even when the install cell already cloned the repo. It checks out `MIMIC_ARM_REF` so the scripts match the name you set.


In [ ]:
import os
import subprocess
from pathlib import Path

# Which branch or tag to download.
# "main" is the current project. To reproduce the v1 baseline, change this
# to "v1-baseline". The environment variable MIMIC_ARM_REF overrides this line.
MIMIC_ARM_REF = "main"
ref = os.environ.get("MIMIC_ARM_REF", MIMIC_ARM_REF)

repo = os.environ.get("MIMIC_ARM_REPO", "https://github.com/Barbodmz/mimic-arm")


def fetch_ref(ref: str) -> str:
    """Download a tag or branch and return the commit id git fetched."""
    tag = subprocess.run(
        ["git", "fetch", "--depth", "1", "origin", "tag", ref],
        capture_output=True,
        text=True,
    )
    if tag.returncode != 0:
        subprocess.check_call(["git", "fetch", "--depth", "1", "origin", ref])
    return subprocess.check_output(["git", "rev-parse", "FETCH_HEAD"], text=True).strip()


if not Path("train.py").is_file():
    subprocess.check_call(
        ["git", "clone", "--depth", "1", "--branch", ref, repo, "mimic-arm"]
    )
    os.chdir("mimic-arm")
else:
    in_git = subprocess.run(
        ["git", "rev-parse", "--is-inside-work-tree"],
        capture_output=True,
        text=True,
    )
    if in_git.returncode == 0 and in_git.stdout.strip() == "true":
        # The install cell may already have cloned main. Switch if needed.
        wanted = fetch_ref(ref)
        current = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
        if current != wanted:
            subprocess.check_call(["git", "checkout", "--detach", wanted])
    else:
        print("train.py is here, but this folder is not a git checkout.")
        print("MIMIC_ARM_REF was not applied.")

head = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True)
print("Working directory:", Path.cwd())
print("MIMIC_ARM_REF:", ref)
if head.returncode == 0:
    print("Checked out:", head.stdout.strip())
print("train.py", Path("train.py").is_file(), "evaluate.py", Path("evaluate.py").is_file())


## Keep checkpoints on Google Drive (optional)

A free Colab runtime deletes `outputs/train` when it disconnects. This cell can mount Google Drive and make `outputs/train` a shortcut to a folder on Drive, so the checkpoints are still there in a new session.

Leave `SAVE_TO_DRIVE = False` if you do not want to connect Drive. The rest of the notebook still works. Checkpoints just live on this VM until the runtime is deleted.

If you set it to `True`, run this cell **before** the Train cell. After a disconnect, start a fresh session, run Install, Get the training scripts, and this cell again, then use **Train longer**. `train.py --resume` finds the newest numbered checkpoint even when Drive could not store LeRobot's `checkpoints/last` shortcut.


In [ ]:
# Checkpoints are written under outputs/train.
# False: they stay on this VM and disappear if Colab disconnects.
# True: mount Google Drive and point outputs/train at a folder there.
# Run this cell before Train. In a new session, run it again before Train longer.
SAVE_TO_DRIVE = False

from pathlib import Path
import shutil

local_train = Path("outputs/train")

if not SAVE_TO_DRIVE:
    print("SAVE_TO_DRIVE is False.")
    print("Checkpoints stay in outputs/train on this VM.")
    print("Set SAVE_TO_DRIVE = True and re-run this cell if you want them on Google Drive.")
else:
    from google.colab import drive

    drive.mount("/content/drive")
    drive_train = Path("/content/drive/MyDrive/mimic-arm/outputs/train")
    drive_train.mkdir(parents=True, exist_ok=True)
    local_train.parent.mkdir(parents=True, exist_ok=True)

    if local_train.is_symlink():
        try:
            already = local_train.resolve() == drive_train.resolve()
        except OSError:
            already = False
        if not already:
            local_train.unlink()
            local_train.symlink_to(drive_train, target_is_directory=True)
    elif local_train.is_dir():
        # Training already created a real folder. Move it onto Drive, then
        # put the shortcut back at the same path so later cells do not change.
        for child in list(local_train.iterdir()):
            destination = drive_train / child.name
            if destination.exists():
                raise SystemExit(
                    f"{destination} already exists on Drive. "
                    "Rename that folder on Drive, then re-run this cell."
                )
            shutil.move(str(child), str(destination))
        local_train.rmdir()
        local_train.symlink_to(drive_train, target_is_directory=True)
    elif local_train.exists():
        raise SystemExit(f"{local_train} exists and is not a folder.")
    else:
        local_train.symlink_to(drive_train, target_is_directory=True)

    print("outputs/train ->", drive_train)
    print("After a disconnect: new session, Install, this cell, then Train longer.")


## Train

This is `python train.py` with the knobs that matter on a free T4:

- `--steps 8000` instead of the default 100,000
- `--batch-size 8`, which is LeRobot's ACT default and fits a T4 for this one-camera task. If the runtime dies with a CUDA out-of-memory error, change it to 4
- `--save_freq=2000` so you can resume
- `--env_eval_freq=0` so training does not spend its time rolling out 50 episodes in the simulator. The next cell does a 20-episode eval on purpose
- `--num-workers 2` to leave RAM for the GPU copy of the batch

The loss log every 200 steps is the thing to read. It should trend down. It will not reach zero.


In [ ]:
import subprocess
import sys

# Same flags as a local `python train.py ...` call. Anything train.py does not
# recognize is forwarded to lerobot-train.
subprocess.check_call(
    [
        sys.executable, "train.py",
        "--steps", "8000",
        "--batch-size", "8",
        "--device", "cuda",
        "--num-workers", "2",
        "--output-dir", "outputs/train/act_aloha_transfer_cube_colab",
        "--save_freq=2000",
        "--env_eval_freq=0",
        "--log_freq=200",
    ]
)


## Evaluate 20 episodes

`evaluate.py` loads `checkpoints/last/pretrained_model` and runs 20 episodes of `AlohaTransferCube`.

Success means the left gripper ended an episode holding the cube off the table (reward 4). At 8,000 steps the success rate can be anywhere from low to pretty good, and 20 episodes is a noisy measurement. A failed episode often still touches the cube. That shows up as max reward 1: contact, but the cube was not lifted.

`--save-failures` writes a video for every failure and one success, with names like `episode_07_fail_maxreward1.mp4` and `episode_03_success.mp4`. The script also prints each episode and a count of how far the runs got:

- 0 nothing
- 1 touched
- 2 lifted
- 3 both grippers
- 4 success

The same table is stored in `eval_info.json`.


In [ ]:
import subprocess
import sys

subprocess.check_call(
    [
        sys.executable, "evaluate.py",
        "--checkpoint", "outputs/train/act_aloha_transfer_cube_colab/checkpoints/last/pretrained_model",
        "--episodes", "20",
        "--save-failures",
        "--device", "cuda",
        "--output-dir", "outputs/eval/act_aloha_transfer_cube_colab",
    ]
)


## Watch a rollout

The video below is one failed episode if the eval saved any, otherwise the first mp4. A name like `episode_07_fail_maxreward1` means that episode touched the cube (reward 1) and did not finish the hand-off. The other videos are in `outputs/eval/act_aloha_transfer_cube_colab/videos/`, including one success when the policy managed a full hand-off.


In [ ]:
from pathlib import Path
from IPython.display import Video, display

videos = sorted(Path("outputs/eval/act_aloha_transfer_cube_colab/videos").glob("*.mp4"))
if not videos:
    videos = sorted(Path("outputs/eval/act_aloha_transfer_cube_colab").rglob("*.mp4"))
if not videos:
    raise SystemExit("No mp4 found. Check the evaluate.py log above.")
failures = [path for path in videos if "_fail_" in path.name]
shown = failures[0] if failures else videos[0]
print(shown)
display(Video(str(shown), embed=True))


## Look inside the network

ACT is one network made of a few blocks. This cell loads the checkpoint you just trained and counts the parameters (the numbers the optimizer changes) in each block. The names are the real ones from LeRobot 0.6.1's `ACTPolicy`:

- `model.backbone` is the ResNet that reads the camera image
- `model.encoder` and `model.decoder` are the transformer that turns that image, plus the joint positions, into an action chunk
- `model.vae_encoder` is the second transformer, used only during training
- the projection layers change the size of a vector on the way in or out

It also prints the settings LeRobot chose: how wide the transformer is (`dim_model`), how many attention heads, how long an action chunk is, and so on. Counting does not need the GPU, so this loads the weights on CPU.


In [ ]:
from mimic_arm.act_summary import print_act_summary

# checkpoints/last is the newest save from the Train cell (or from Train longer,
# if you already ran that). The counts do not change with more training steps.
print_act_summary(
    "outputs/train/act_aloha_transfer_cube_colab/checkpoints/last/pretrained_model",
    device="cpu",
)


## Train longer (resume)

8,000 steps is a start. This cell keeps training until `TOTAL_STEPS`.

`TOTAL_STEPS` is the new finish line, not "how many extra steps to add". If the checkpoint is at step 8,000 and `TOTAL_STEPS` is 20,000, training does 12,000 more steps and stops. It does not start over at step 0.

Use the same output directory, batch size, and device as the first Train cell. LeRobot reads the step counter, the optimizer, and the data order out of the checkpoint.

If this is a new Colab session, run Install, Get the training scripts, and the Drive cell (when you used it) before this one.


In [ ]:
import subprocess
import sys

# New finish line. The checkpoint remembers the step it already reached.
TOTAL_STEPS = 20000

subprocess.check_call(
    [
        sys.executable, "train.py",
        "--resume",
        "--steps", str(TOTAL_STEPS),
        "--batch-size", "8",
        "--device", "cuda",
        "--num-workers", "2",
        "--output-dir", "outputs/train/act_aloha_transfer_cube_colab",
        "--save_freq=2000",
        "--env_eval_freq=0",
        "--log_freq=200",
    ]
)


## Evaluate 50 episodes and watch the failures

Fifty episodes is still a noisy score (the compare cell prints the error bars), but it is a wider look than 20. `--save-failures` keeps every failed episode and one success.

The max reward of an episode says how far the hand-off got:

- 0 the arms never got the cube
- 1 a gripper touched the cube
- 2 the cube was lifted
- 3 both grippers had it
- 4 success: the left gripper is holding it off the table


In [ ]:
import subprocess
import sys

subprocess.check_call(
    [
        sys.executable, "evaluate.py",
        "--checkpoint", "outputs/train/act_aloha_transfer_cube_colab/checkpoints/last/pretrained_model",
        "--episodes", "50",
        "--save-failures",
        "--device", "cuda",
        "--output-dir", "outputs/eval/act_aloha_transfer_cube_colab_50",
    ]
)


## Failure videos

Each clip below is one episode that did not succeed, then the one success video if there is one. Read the file name. `episode_07_fail_maxreward1.mp4` means episode 7 touched the cube and did not lift it.


In [ ]:
from pathlib import Path
from IPython.display import Video, display

folder = Path("outputs/eval/act_aloha_transfer_cube_colab_50/videos")
videos = sorted(folder.glob("*.mp4"))
if not videos:
    raise SystemExit(f"No mp4 files in {folder}. Check the evaluate log above.")
failures = [path for path in videos if "_fail_" in path.name]
successes = [path for path in videos if "_success" in path.name]
print(f"{len(failures)} failure video(s), {len(successes)} success video(s)")
for path in failures + successes:
    print(path.name)
    display(Video(str(path), embed=True))


## Compare checkpoints

Training saved a checkpoint every 2,000 steps. This cell scores each numbered checkpoint on the same 20 episodes (the same seeds), then prints success rate, average max reward, and a 95% Wilson interval.

That interval is the honest part. 20 episodes is a small sample, so the range on each row is wide. If two rows' intervals overlap, you cannot tell them apart yet. Change `--episodes` to `50` for a tighter range. It takes longer.

You can run this before **Train longer**. It scores whatever numbered checkpoints are already on disk. The table is also saved as `outputs/eval/compare_colab/compare.csv`.


In [ ]:
import subprocess
import sys

subprocess.check_call(
    [
        sys.executable, "compare.py",
        "--train-dir", "outputs/train/act_aloha_transfer_cube_colab",
        "--episodes", "20",
        "--device", "cuda",
        "--output-dir", "outputs/eval/compare_colab",
    ]
)
